In [ ]:
! pip install transformers accelerate torch

In [ ]:
# One-cell pipeline: Qwen 3.5 generates/refines; Qwen 2.5 evaluates.

import gc
import json
import os
import re

import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

GENERATOR_MODEL_ID = "Qwen/Qwen3.5-4B"
# JUDGE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" # new model
SEED_INPUT_PATH = "/kaggle/input/datasets/shauryathakur007/fineweb/fineweb_sample.jsonl"
INITIAL_OUTPUT_PATH = "initial_file_instruction.jsonl"
REGENERATED_OUTPUT_TEMPLATE = "regenerated_instruction_{attempt}.jsonl"
MURI_SOURCE = "muri-train"
SCORE_THRESHOLD = 3
MAX_REGENERATION_ATTEMPTS = 3
DTYPE = torch.float16

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab: Runtime -> Change runtime type -> GPU.")


INITIAL_PROMPT = """Utgå från den svenska källtexten nedan och skriv EN enda naturlig svensk fråga eller instruktion som texten kan vara ett korrekt svar på.

[KÄLLTEXT]
{seed_doc}

Hårda formatkrav:
- Svara med exakt en rad: endast den färdiga frågan eller instruktionen.
- Ingen analys, motivering, rubrik, punktlista, citattecken eller etikett.
- Skriv inte "Thinking Process", "Analys", "Svar" eller liknande.
- Använd idiomatisk svenska och lägg inte till fakta som saknas i källtexten.
"""

REFINEMENT_PROMPT = """Skriv om den tidigare svenska instruktionen så att den åtgärdar utvärderarens återkoppling.

[KÄLLTEXT]
{seed_doc}

[TIDIGARE INSTRUKTION]
{instruction}

[UTVÄRDERARENS ÅTERKOPPLING]
{eval_reasoning}

Svara med exakt en rad och endast den förbättrade frågan eller instruktionen på idiomatisk svenska.
Ingen analys, motivering, rubrik, punktlista eller etikett. Lägg inte till fakta som saknas i källtexten.
"""

JUDGE_PROMPT = """Du är en sträng kvalitetsgranskare av svenska instruktioner.

[KÄLLTEXT / FÖRVÄNTAT SVAR]
{seed_doc}

[GENERERAD INSTRUKTION]
{instruction}

Bedöm på en skala 1–5. En femma ska vara ovanlig och ges endast när båda kraven är helt uppfyllda.

- relevance_score: 5 endast om källtexten är ett mycket naturligt, direkt och korrekt svar på just instruktionen. 3 betyder delvis relevant; 1 betyder fel, alltför vag eller metatext.
- idiomaticity_score: 5 endast om texten är en enda välformulerad, idiomatisk svensk fråga/instruktion. Sätt 1 om den innehåller resonemang, analys, rubriker, punktlistor, formatinstruktioner, engelska metauttryck eller flera steg i stället för själva instruktionen.

Svara endast med giltig JSON:
{{"relevance_score": <heltal 1-5>, "idiomaticity_score": <heltal 1-5>, "reasoning": "kort motivering på svenska"}}
"""


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE, device_map="auto")
    model.eval()
    return tokenizer, model


def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


def make_inputs(tokenizer, model, prompt):
    messages = [
        {"role": "system", "content": "Ge endast det efterfrågade svaret. Visa aldrig tankegång, analys eller arbetssteg."},
        {"role": "user", "content": prompt},
    ]
    kwargs = dict(
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    )
    # Qwen 3.5 templates support disabling visible reasoning. Older templates safely fall back.
    try:
        inputs = tokenizer.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        inputs = tokenizer.apply_chat_template(messages, **kwargs)
    return inputs.to(model.device)


def normalize_instruction(text):
    text = text.strip()
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return re.sub(r"\s+", " ", text).strip().strip('"')


def is_valid_instruction(instruction):
    if not instruction or len(instruction) > 350 or "\n" in instruction:
        return False
    forbidden = ("thinking process", "analysis:", "analys:", "step by step", "<think>", "1.", "**", "input:", "output:")
    request_starts = ("vad ", "vem ", "vilken ", "vilket ", "vilka ", "hur ", "var ", "när ", "beskriv ", "skriv ", "förklara ", "redogör ", "återge ", "berätta ", "ge ", "identifiera ", "sammanfatta ")
    return not any(marker in instruction.lower() for marker in forbidden) and ("?" in instruction or instruction.lower().startswith(request_starts))


def generate_instruction(tokenizer, model, prompt):
    inputs = make_inputs(tokenizer, model, prompt)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.35,
            top_p=0.85,
            repetition_penalty=1.08,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    return normalize_instruction(tokenizer.decode(generated_ids, skip_special_tokens=True))


def judge_instruction(tokenizer, model, seed_doc, instruction):
    # Invalid format receives deterministic failing scores; it must never be rewarded by the LLM judge.
    if not is_valid_instruction(instruction):
        return {
            "relevance_score": 1,
            "idiomaticity_score": 1,
            "reasoning": "Instruktionen är inte en ensam, färdig svensk fråga eller instruktion utan innehåller metatext eller resonemang.",
        }

    inputs = make_inputs(tokenizer, model, JUDGE_PROMPT.format(seed_doc=seed_doc, instruction=instruction))
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=160, do_sample=False)
    response = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    try:
        if "```" in response:
            response = response.split("```")[1].replace("json", "", 1).strip()
        result = json.loads(response)
        relevance = result.get("relevance_score")
        idiomaticity = result.get("idiomaticity_score")
        if not isinstance(relevance, int) or not isinstance(idiomaticity, int):
            raise ValueError("Scores must be integers")
        if not (1 <= relevance <= 5 and 1 <= idiomaticity <= 5):
            raise ValueError("Scores must be from 1 to 5")
        return {"relevance_score": relevance, "idiomaticity_score": idiomaticity, "reasoning": str(result.get("reasoning", ""))}
    except (json.JSONDecodeError, ValueError, TypeError, IndexError) as error:
        return {"relevance_score": 1, "idiomaticity_score": 1, "reasoning": f"Kunde inte tolka domarens svar: {error}"}


def needs_refinement(evaluation):
    return evaluation["relevance_score"] < SCORE_THRESHOLD or evaluation["idiomaticity_score"] < SCORE_THRESHOLD


def passes_threshold(evaluation):
    return evaluation["relevance_score"] >= SCORE_THRESHOLD and evaluation["idiomaticity_score"] >= SCORE_THRESHOLD


if not os.path.exists(SEED_INPUT_PATH):
    raise FileNotFoundError(f"Seed file not found: {SEED_INPUT_PATH}")
with open(SEED_INPUT_PATH, "r", encoding="utf-8") as f:
    seeds = [json.loads(line) for line in f if line.strip()]

# FineWeb seed rows use `text`; retain their original fields only during processing.
# Output IDs are assigned in the requested MURI training-set convention.
records = [
    {
        "id": f"{MURI_SOURCE}-{row_no}",
        "source": MURI_SOURCE,
        "seed_doc": seed["text"].strip(),
    }
    for row_no, seed in enumerate(seeds)
    if seed.get("text", "").strip()
]


def evaluation_for_output(evaluation):
    return {
        "relevance": evaluation["relevance_score"],
        "idiomaticity": evaluation["idiomaticity_score"],
        "reasoning": evaluation["reasoning"],
    }


def muri_record(record):
    return {
        "id": record["id"],
        "source": record["source"],
        "messages": [
            {"role": "user", "content": record["current_instruction"]},
            {"role": "assistant", "content": record["seed_doc"]},
        ],
    }


def write_snapshot(output_path, include_evaluation=True):
    """Write all records in MURI message format; evaluation is excluded only from the final snapshot."""
    with open(output_path, "w", encoding="utf-8") as f:
        for record in records:
            output_record = muri_record(record)
            if include_evaluation:
                output_record["evaluation"] = evaluation_for_output(record["current_evaluation"])
                output_record["retries"] = record["retries"]
            f.write(json.dumps(output_record, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to: {output_path}")


# Stage 0: initial generation and evaluation. This checkpoint is MURI-shaped plus evaluation metadata.
generator_tokenizer, generator_model = load_model(GENERATOR_MODEL_ID)
for record in tqdm(records, desc="Generating initial instructions with Qwen 3.5"):
    record["current_instruction"] = generate_instruction(
        generator_tokenizer, generator_model, INITIAL_PROMPT.format(seed_doc=record["seed_doc"])
    )
release_model(generator_model)
del generator_model, generator_tokenizer
gc.collect()
torch.cuda.empty_cache()

judge_tokenizer, judge_model = load_model(JUDGE_MODEL_ID)
for record in tqdm(records, desc="Evaluating initial instructions with Qwen 2.5"):
    record["current_evaluation"] = judge_instruction(
        judge_tokenizer, judge_model, record["seed_doc"], record["current_instruction"]
    )
    record["retries"] = []
release_model(judge_model)
del judge_model, judge_tokenizer
gc.collect()
torch.cuda.empty_cache()

write_snapshot(INITIAL_OUTPUT_PATH, include_evaluation=True)


# Stages 1–3: regenerate only rows with relevance OR idiomaticity below 3.
# `regenerated_instruction_3.jsonl` is the final strict-MURI file, with no evaluation fields.
for attempt in range(1, MAX_REGENERATION_ATTEMPTS + 1):
    candidates = [record for record in records if needs_refinement(record["current_evaluation"])]

    if candidates:
        generator_tokenizer, generator_model = load_model(GENERATOR_MODEL_ID)
        for record in tqdm(candidates, desc=f"Regenerating retry {attempt} with Qwen 3.5"):
            previous_evaluation = record["current_evaluation"]
            record["current_instruction"] = generate_instruction(
                generator_tokenizer,
                generator_model,
                REFINEMENT_PROMPT.format(
                    seed_doc=record["seed_doc"],
                    instruction=record["current_instruction"],
                    eval_reasoning=previous_evaluation["reasoning"],
                ),
            )
            record["pending_feedback"] = previous_evaluation["reasoning"]
        release_model(generator_model)
        del generator_model, generator_tokenizer
        gc.collect()
        torch.cuda.empty_cache()

        # Evaluation remains available internally for the final pass, but is omitted from its file.
        judge_tokenizer, judge_model = load_model(JUDGE_MODEL_ID)
        for record in tqdm(candidates, desc=f"Evaluating retry {attempt} with Qwen 2.5"):
            retry_evaluation = judge_instruction(
                judge_tokenizer, judge_model, record["seed_doc"], record["current_instruction"]
            )
            record["retries"].append({
                "attempt": attempt,
                "feedback": record["pending_feedback"],
                "evaluation": evaluation_for_output(retry_evaluation),
            })
            record["current_evaluation"] = retry_evaluation
            del record["pending_feedback"]
        release_model(judge_model)
        del judge_model, judge_tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print(f"No rows require regeneration before retry {attempt}; carrying the prior snapshot forward.")

    is_final_stage = attempt == MAX_REGENERATION_ATTEMPTS
    write_snapshot(
        REGENERATED_OUTPUT_TEMPLATE.format(attempt=attempt),
        include_evaluation=not is_final_stage,
    )


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (36 outputs left out)

Regenerating retry 3 with Qwen 3.5: 100%|██████████| 25/25 [01:22<00:00,  3.31s/it]


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Evaluating retry 3 with Qwen 2.5: 100%|██████████| 25/25 [02:40<00:00,  6.43s/it]


Saved 100 records to: regenerated_instruction_3.jsonl
